In [0]:
import sys
sys.path.append('..')
sys.path.append('../..')
import lib_dna_member.generate_population as gp
import lib_dna_member.job_manager as managers
import lib_dna_member.most_shopped_features as features
from  lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:

def generate_most_shopped(job):
    """
    Generate first and second most shopped category for each memberber in the
    given population
    Parameters:
        job (object): Job Manager object based on the current config file

    Returns:
        (pyspark.sql.DataFrame): Transaction data of the given population
    """
    # dna = job.data.tables["population"]
    dna = job.tables["population"]
    orig_cols = dna.columns

    num_weeks = ["FIFTY-TWO"]

    dna = features.feature_most_shopped_category(job, dna, num_weeks)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna


In [0]:


def main():
    job = managers.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

    recency_lookback_duration = job.config["params"].get("recency_lookback_duration", {})

    member_dna_input_data_validator(
        silver_transaction_fiscal_detail,
        silver_master_member_extended,
        silver_skeleton,
        silver_bcg_maps_ah5_customer_facing_desc,
        recency_lookback_duration=recency_lookback_duration,
        spark=spark
    )   


    # member_dna_input_data_validator(
        # job,
        # recency_lookback_duration,
        # [
            # "detail_path",
            # "member_extended_path",
            # "skeleton_path",
            # "AH5_custumer_facing_desc_path",
        # ],
    # )

    job.read_table("detail_fiscal")
    job.read_table("member_extended")
    job.read_table("skeleton")
    job.read_table("AH5_custumer_facing_desc")

    job.tables["detail_fiscal"] = gp.apply_fw_date_range(
        job, job.tables["detail_fiscal"]
    )

    population = gp.generate_population(job)
    job.tables["population"] = population

    feature_population = gp.generate_population(job, "feature")
    job.tables["feature_population"] = feature_population

    features = generate_most_shopped(job)
    features = features.withColumn('MBRSHP_SID', f.coalesce('MBRSHP_SID', f.lit(-1)))

    spark.sql(f"DELETE FROM {fs_cubes_most_shopped}")

    fe = FeatureEngineeringClient()

    fe.write_table(
        name=fs_cubes_most_shopped,
        df=features,
        mode="merge"
    )




In [0]:

if __name__ == "__main__":
    main()